# OASIS Infobyte Data Analytics Internship
## Task 3: Data Cleaning

### Introduction
Data cleaning is the process of identifying and correcting errors, missing values, duplicate records, and inconsistent information in a dataset.

### Objective
The objective of this project is to clean the Cafe Sales dataset, improve its data quality, identify missing values, and prepare the data for further analysis.

### Dataset
The dataset used for this project is dirty_cafe_sales.csv.

In [1]:
import pandas as pd
import numpy as np
from google.colab import files

### Step 1: Loading the Dataset

The original Cafe Sales dataset is uploaded into Google Colab so that it can be inspected and cleaned using Python.

In [6]:
uploaded = files.upload()

Saving dirty_cafe_sales.csv to dirty_cafe_sales.csv


### Step 2: Reading the Dataset

The dataset is loaded into a pandas DataFrame. A DataFrame is a table containing rows and columns, similar to an Excel worksheet.

In [7]:
df = pd.read_csv("dirty_cafe_sales.csv")

print("Dataset loaded successfully!")
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

df.head()

Dataset loaded successfully!
Number of rows: 10000
Number of columns: 8


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,ERROR,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,UNKNOWN,UNKNOWN,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Step 3: Exploring the Dataset

The dataset is inspected to understand its structure, identify the column names, check the data types, and find missing values and duplicate records.

In [8]:
print("Column names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

print("\nMissing values in each column:")
print(df.isnull().sum())

print("\nNumber of duplicate rows:")
print(df.duplicated().sum())

Column names:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Transaction ID    10000 non-null  object
 1   Item              9667 non-null   object
 2   Quantity          9862 non-null   object
 3   Price Per Unit    9821 non-null   object
 4   Total Spent       9827 non-null   object
 5   Payment Method    7421 non-null   object
 6   Location          6735 non-null   object
 7   Transaction Date  9841 non-null   object
dtypes: object(8)
memory usage: 625.1+ KB

Missing values in each column:
Transaction ID         0
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
d

### Step 4: Creating a Working Copy

A copy of the original dataset is created before cleaning. This protects the original data from being changed during the cleaning process.

In [9]:
clean_df = df.copy()

print("Working copy created successfully!")

Working copy created successfully!


### Step 5: Cleaning Text Values

Extra spaces are removed from column names and text values. Common placeholder values, such as ERROR and UNKNOWN, are identified and treated as missing values.

In [10]:
# Remove extra spaces from column names
clean_df.columns = clean_df.columns.str.strip()

# Clean text values
text_columns = clean_df.select_dtypes(
    include=["object", "string"]
).columns

for col in text_columns:
    clean_df[col] = clean_df[col].astype("string").str.strip()

# Replace common invalid markers with missing values
missing_markers = [
    "", "ERROR", "UNKNOWN", "N/A", "NA", "NULL", "NONE"
]

for col in text_columns:
    clean_df[col] = clean_df[col].replace(
        missing_markers, pd.NA
    )

print("Text cleaning completed!")

clean_df.head()

Text cleaning completed!


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,<NA>,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,<NA>,<NA>,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Step 6: Correcting Data Types

Numeric columns are converted to numeric data types, while the transaction date column is converted to a date format. Invalid entries are converted to missing values.

In [11]:
# Convert numeric columns
numeric_columns = [
    "Quantity",
    "Price Per Unit",
    "Total Spent"
]

for col in numeric_columns:
    if col in clean_df.columns:
        clean_df[col] = pd.to_numeric(
            clean_df[col], errors="coerce"
        )

# Convert transaction date
if "Transaction Date" in clean_df.columns:
    clean_df["Transaction Date"] = pd.to_datetime(
        clean_df["Transaction Date"],
        errors="coerce"
    )

print("Data type conversion completed!")

clean_df.info()

Data type conversion completed!
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction ID    10000 non-null  string        
 1   Item              9031 non-null   string        
 2   Quantity          9521 non-null   Int64         
 3   Price Per Unit    9467 non-null   Float64       
 4   Total Spent       9498 non-null   Float64       
 5   Payment Method    6822 non-null   string        
 6   Location          6039 non-null   string        
 7   Transaction Date  9540 non-null   datetime64[ns]
dtypes: Float64(2), Int64(1), datetime64[ns](1), string(4)
memory usage: 654.4 KB


### Step 7: Handling Missing Values

Missing values are examined and corrected where possible. For example, Total Spent can be calculated by multiplying Quantity by Price Per Unit when both values are available.

Values that cannot be reliably recovered are left missing rather than guessed.

In [12]:
# Calculate missing Total Spent values
if all(col in clean_df.columns for col in [
    "Quantity", "Price Per Unit", "Total Spent"
]):

    mask = (
        clean_df["Total Spent"].isna()
        & clean_df["Quantity"].notna()
        & clean_df["Price Per Unit"].notna()
    )

    clean_df.loc[mask, "Total Spent"] = (
        clean_df.loc[mask, "Quantity"]
        * clean_df.loc[mask, "Price Per Unit"]
    )

print("Missing Total Spent values corrected where possible.")

print("\nRemaining missing values:")
print(clean_df.isnull().sum())

Missing Total Spent values corrected where possible.

Remaining missing values:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent           40
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64


### Step 8: Removing Duplicate Records

Duplicate rows are identified and removed to reduce repeated records in the dataset.

In [13]:
duplicates_before = clean_df.duplicated().sum()

clean_df = clean_df.drop_duplicates()

duplicates_after = clean_df.duplicated().sum()

print("Duplicate rows found:", duplicates_before)
print("Duplicate rows remaining:", duplicates_after)
print("Duplicate removal completed!")

Duplicate rows found: 0
Duplicate rows remaining: 0
Duplicate removal completed!


### Step 9: Validating the Cleaned Dataset

The cleaned dataset is inspected to confirm the number of rows and columns, check remaining missing values, and review the cleaned records.

In [14]:
print("Original dataset shape:", df.shape)
print("Cleaned dataset shape:", clean_df.shape)

print("\nMissing values after cleaning:")
print(clean_df.isnull().sum())

print("\nFirst five rows of the cleaned dataset:")
clean_df.head()

Original dataset shape: (10000, 8)
Cleaned dataset shape: (10000, 8)

Missing values after cleaning:
Transaction ID         0
Item                 969
Quantity             479
Price Per Unit       533
Total Spent           40
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

First five rows of the cleaned dataset:


,Transaction ID,Item,Quantity,Price Per Unit,Total Spent,Payment Method,Location,Transaction Date
0,TXN_1961373,Coffee,2,2.0,4.0,Credit Card,Takeaway,2023-09-08
1,TXN_4977031,Cake,4,3.0,12.0,Cash,In-store,2023-05-16
2,TXN_4271903,Cookie,4,1.0,4.0,Credit Card,In-store,2023-07-19
3,TXN_7034554,Salad,2,5.0,10.0,<NA>,<NA>,2023-04-27
4,TXN_3160411,Coffee,2,2.0,4.0,Digital Wallet,In-store,2023-06-11


### Step 10: Data Quality Summary

A summary is created to show the number and percentage of missing values remaining in each column after cleaning.

In [15]:
quality_summary = pd.DataFrame({
    "Column": clean_df.columns,
    "Missing Values": [
        clean_df[col].isnull().sum()
        for col in clean_df.columns
    ],
    "Missing Percentage": [
        round(clean_df[col].isnull().mean() * 100, 2)
        for col in clean_df.columns
    ]
})

quality_summary

,Column,Missing Values,Missing Percentage
0,Transaction ID,0,0.00
1,Item,969,9.69
2,Quantity,479,4.79
3,Price Per Unit,533,5.33
4,Total Spent,40,0.40
5,Payment Method,3178,31.78
6,Location,3961,39.61
7,Transaction Date,460,4.60


### Step 11: Exporting the Cleaned Dataset

The cleaned dataset and data quality summary are saved as CSV files. These files can be downloaded and included in the project submission.

In [16]:
# Save the cleaned dataset
clean_df.to_csv("cleaned_cafe_sales.csv", index=False)

# Save the data quality summary
quality_summary.to_csv(
    "cafe_data_quality_summary.csv",
    index=False
)

print("Files saved successfully!")

Files saved successfully!


In [17]:
files.download("cleaned_cafe_sales.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
files.download("cafe_data_quality_summary.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Conclusion

The Cafe Sales dataset was inspected and cleaned using Python. The cleaning process included standardizing text values, correcting data types, handling recoverable missing values, removing duplicate records, and checking the quality of the cleaned data.

A cleaned dataset and a data quality summary were generated for further analysis.

This project demonstrates how Python can be used to improve data quality and prepare raw data for analysis.